# KPI Framework

The KPI framework translates exploratory analysis findings into measurable business metrics that can be monitored through dashboards. The selected KPIs focus on market size, pricing, customer satisfaction, host performance, and review activity.

In [1]:
import pandas as pd
import numpy as np

In [2]:
dim_listing = pd.read_csv(
    "../data/processed/dim_listing.csv"
)

dim_host = pd.read_csv(
    "../data/processed/dim_host.csv",
    parse_dates=["host_since"],
    low_memory=False
)

dim_location = pd.read_csv(
    "../data/processed/dim_location.csv"
)

fact_reviews = pd.read_csv(
    "../data/processed/fact_reviews.csv",
    parse_dates=["date"]
)

In [3]:
print("Listings :", len(dim_listing))
print("Hosts    :", len(dim_host))
print("Locations:", len(dim_location))
print("Reviews  :", len(fact_reviews))

Listings : 279434
Hosts    : 181805
Locations: 660
Reviews  : 5372983


## KPI Category 1: Market Size & Supply

In [4]:
market_kpis = pd.DataFrame({
    "KPI": [
        "Total Listings",
        "Total Hosts",
        "Total Cities",
        "Total Neighbourhoods"
    ],
    "Value": [
        len(dim_listing),
        dim_host["host_id"].nunique(),
        dim_location["city"].nunique(),
        dim_location["neighbourhood"].nunique()
    ]
})

market_kpis

,KPI,Value
0,Total Listings,279434
1,Total Hosts,181805
2,Total Cities,10
3,Total Neighbourhoods,660


### Market Size & Supply KPI Definitions

| KPI | Business Meaning | Formula |
|------|------|------|
| Total Listings | Total Airbnb properties available in the dataset | Count of Listing IDs |
| Total Hosts | Total unique hosts operating on Airbnb | Count of Unique Host IDs |
| Total Cities | Number of cities included in the analysis | Count of Unique Cities |
| Total Neighbourhoods | Number of neighbourhood markets covered | Count of Unique Neighbourhoods |

## KPI Category 2: Pricing Intelligence

### Pricing Intelligence KPI Definitions

| KPI | Business Meaning | Formula |
|------|------|------|
| Median Listing Price | Typical listing price after reducing outlier impact | Median Price |
| Average Listing Price | Overall average listing price | Mean Price |
| Median Bedrooms | Typical bedroom count per listing | Median Bedrooms |
| Median Accommodation Capacity | Typical guest capacity per listing | Median Accommodates |

In [5]:
pricing_kpis = pd.DataFrame({
    "KPI": [
        "Median Listing Price",
        "Average Listing Price",
        "Median Bedrooms",
        "Median Accommodation Capacity"
    ],
    "Value": [
        round(dim_listing["price"].median(), 2),
        round(dim_listing["price"].mean(), 2),
        dim_listing["bedrooms"].median(),
        dim_listing["accommodates"].median()
    ]
})

pricing_kpis

,KPI,Value
0,Median Listing Price,150.00
1,Average Listing Price,609.13
2,Median Bedrooms,1.00
3,Median Accommodation Capacity,2.00


#### KPI Insights

Median Listing Price is the primary pricing KPI because the Airbnb dataset contains substantial price outliers that significantly inflate the average price.

Median Bedrooms and Median Accommodation Capacity represent the typical property profile available across the Airbnb marketplace and were identified during exploratory analysis as important pricing drivers.

## KPI Category 3: Demand Intelligence

### Demand Intelligence KPI Definitions

| KPI | Business Meaning | Formula |
|------|------|------|
| Total Reviews | Total customer reviews recorded | Count of Review IDs |
| Average Reviews per Listing | Average customer engagement per listing | Total Reviews / Total Listings |
| Peak Review Year | Year with highest review activity | Year with Maximum Reviews |
| Peak Review Month | Month with highest review activity | Month with Maximum Reviews |

In [6]:
total_reviews = len(fact_reviews)

avg_reviews_per_listing = round(
    total_reviews / len(dim_listing),
    2
)

peak_review_year = (
    fact_reviews["date"]
    .dt.year
    .value_counts()
    .idxmax()
)

month_map = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December"
}

peak_review_month = month_map[
    fact_reviews["date"]
    .dt.month
    .value_counts()
    .idxmax()
]

demand_kpis = pd.DataFrame({
    "KPI": [
        "Total Reviews",
        "Average Reviews per Listing",
        "Peak Review Year",
        "Peak Review Month"
    ],
    "Value": [
        total_reviews,
        avg_reviews_per_listing,
        peak_review_year,
        peak_review_month
    ]
})

demand_kpis

,KPI,Value
0,Total Reviews,5372983
1,Average Reviews per Listing,19.23
2,Peak Review Year,2019
3,Peak Review Month,October


#### KPI Insights

Review activity serves as a proxy for customer demand and marketplace engagement.

Peak Review Year and Peak Review Month were identified during exploratory analysis as important indicators of long-term growth trends and seasonal demand patterns across Airbnb markets.

## KPI Category 4: Host Intelligence

### Host Intelligence KPI Definitions

| KPI | Business Meaning | Formula |
|------|------|------|
| Superhost Percentage | Percentage of listings operated by Superhosts | Superhost Listings / Total Listings |
| Average Review Rating | Overall customer satisfaction score | Mean Review Rating |
| Superhost Average Rating | Average rating achieved by Superhosts | Mean Rating (Superhosts) |
| Non-Superhost Average Rating | Average rating achieved by non-Superhosts | Mean Rating (Non-Superhosts) |

In [7]:
host_analysis = dim_listing.merge(
    dim_host[
        ["host_id", "host_is_superhost"]
    ],
    on="host_id",
    how="left"
)

superhost_pct = round(
    (
        host_analysis["host_is_superhost"]
        .eq("t")
        .sum()
        / len(host_analysis)
    ) * 100,
    2
)

avg_rating = round(
    dim_listing["review_scores_rating"]
    .mean(),
    2
)

superhost_rating = round(
    host_analysis.loc[
        host_analysis["host_is_superhost"] == "t",
        "review_scores_rating"
    ].mean(),
    2
)

non_superhost_rating = round(
    host_analysis.loc[
        host_analysis["host_is_superhost"] == "f",
        "review_scores_rating"
    ].mean(),
    2
)

host_kpis = pd.DataFrame({
    "KPI": [
        "Superhost Percentage",
        "Average Review Rating",
        "Superhost Average Rating",
        "Non-Superhost Average Rating"
    ],
    "Value": [
        superhost_pct,
        avg_rating,
        superhost_rating,
        non_superhost_rating
    ]
})

host_kpis

,KPI,Value
0,Superhost Percentage,17.98
1,Average Review Rating,93.41
2,Superhost Average Rating,97.00
3,Non-Superhost Average Rating,92.26


#### KPI Insights

Host quality has a measurable impact on customer satisfaction.

Exploratory analysis showed that Superhosts consistently achieve higher review ratings and command higher pricing than non-superhosts. Therefore, Superhost-related KPIs are important indicators of marketplace quality and host performance.

In [8]:
kpi_framework = pd.concat([
    market_kpis.assign(Category="Market Intelligence"),
    pricing_kpis.assign(Category="Pricing Intelligence"),
    demand_kpis.assign(Category="Demand Intelligence"),
    host_kpis.assign(Category="Host Intelligence")
])

kpi_framework = kpi_framework[
    ["Category", "KPI", "Value"]
]

kpi_framework = kpi_framework.reset_index(drop=True)

kpi_framework

,Category,KPI,Value
0,Market Intelligence,Total Listings,279434
1,Market Intelligence,Total Hosts,181805
2,Market Intelligence,Total Cities,10
3,Market Intelligence,Total Neighbourhoods,660
4,Pricing Intelligence,Median Listing Price,150.0
5,Pricing Intelligence,Average Listing Price,609.13
6,Pricing Intelligence,Median Bedrooms,1.0
7,Pricing Intelligence,Median Accommodation Capacity,2.0
8,Demand Intelligence,Total Reviews,5372983
9,Demand Intelligence,Average Reviews per Listing,19.23


## KPI Framework Conclusion

A total of 16 KPIs were selected to support the core objectives of the Airbnb Pricing & Market Intelligence project.

The KPI framework is organized into four business domains:

- Market Intelligence
- Pricing Intelligence
- Demand Intelligence
- Host Intelligence

These KPIs convert exploratory analysis findings into measurable business metrics and provide the foundation for dashboard development. The framework supports monitoring of market structure, pricing dynamics, customer demand patterns, and host performance across Airbnb markets.

The selected KPIs directly align with the project's business objectives, analytical questions, and dashboard reporting requirements.